## **sklearn训练模型工作流程**

使用sklearn创建并训练机器学习模型的范式通常为：

**1. 导入模型**：

from sklearn.xxx import xxx

**2. 数据预处理：数据加载、清洗、预处理、数据集划分**

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

**3. 创建模型**：

model = xxx()

**4. 训练模型**：

model.fit(X_train, y_train)

**5. 模型预测**：

y_pred = model.predict(X_test)

score = accuracy_score(y_test, y_pred)


## **传统分类算法(以鸢尾花数据集的分类为例)**

### **决策树**
根据数据的特征通过对其逐步分裂，以树形结构做出决策，一步步选择，从而将数据分类到不同的类别。

决策树的步骤：
1. **选择一个最佳的特征进行划分**。
   * **核心思想**：通过概率计算，找出能够最好地区分不同类别的数据特征。
   * 选择算法常用的有：ID4, C4.5, CART
  
   * ID4选择的判据是信息增益。
   信息熵：$\mathrm{Ent(D)=-\sum_{i=1}^nP(x_k)\log_2P(x_k)}$
   其中D是样本集合，集合中第k类样本所占比例为$P(x_k)$,假如只有两类，则在两类的概率都取0.5时，信息熵取到最大值。
   
   信息增益：$\mathrm{Gain(D,A)=Ent(D)-\sum_{v=1}^V\frac{D^v}{D}Ent(D^v)}$
   其中D是样本集合，A是特征，$D^v$是特征A取值为v的样本子集，$V$是特征A的取值个数。即用特征A区分前集合D的信息熵减去特征A区分后各小样本空间的加权信息熵。

   * C4.5选择最佳特征的判据是信息增益率。

   信息增益率：$\mathrm{Gain\_ratio(D,A)=\frac{Gain(D,A)}{IV(A)}}$
   其中IV(A)是特征A的固有值。IV(A)=$-\sum_{v=1}^V\frac{|D^v|}{|D|}\log_2\frac{|D^v|}{|D|}$,将特征A的分类数也考虑了进来，避免了为了信息增益的最大化而将特征A无限细分。

   * CART选择最佳特征的判据是基尼指数数。
  
   基尼值度量数据集D的纯度，公式为：$Gini(D)=1-\sum_{k=1}^Kp_k^2$，其中$p_k$是样本集合D中第k类样本所占的比例。基尼值越大表明数据集的纯度越低。

   属性A的基尼指数：$\mathrm{Gini\_index(D,A)=\sum_{v=1}^V\frac{|D^v|}{|D|}Gini(D^v)}$表明了在特征A划分下的（加权）基尼指数，我们要选择的属性A使得属性A的基尼指数最小。

2. **根据这个特征把数据分成不同分支**。
3. **对每个分支重复上述步骤**，直到所有数据被正确分类或没有更多特征可供选择。
4. (可选)**决策树剪枝**。在测试集里划分出一个验证集，来判断剪枝前后识别精确率的变化。
   * **原因**：为了尽可能正确分类训练样本，节点的划分过程会不断重复直到不能再分，这样就可能对训练样本学习的“太好”了，把训练样本的一些特点当做所有数据都具有的一般性质，从而导致过拟合。
   * 预剪枝：在每次划分前后，计算验证集的准确率，当准确率下降时，终止划分。
   * 后剪枝：先将决策树完全构建出来，然后自底向上地对非叶节点进行考察，若将该节点对应的子树替换为叶节点能带来决策树泛化性能的提升，则将该子树替换为叶节点。

In [1]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 加载数据
iris = load_iris()
X, y = iris.data, iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 创建决策树模型

dt = DecisionTreeClassifier(criterion='gini', splitter='best', min_samples_split=5, max_depth=4)
'''
criterion: 用于选择分割节点的标准(默认为"gini"),决定了节点分裂的方式,可以是"gini"(基尼不纯度)或"entropy"(信息增益)。
splitter: 用于选择分割节点的策略(默认为"best"),可以是"best"(最佳分割)或"random"(随机分割)。
max_depth: 树的最大深度(默认为None),用于防止过拟合,None表示节点可以扩展到所有叶节点都是纯的。
min_samples_split: 内部节点再划分所需的最小样本数(默认为2),较小的值会创建更复杂的树。
min_samples_leaf: 叶子节点最少样本数(默认为1),限制了叶子节点的最小尺寸,可以防止过拟合。
max_features: 寻找最佳分割时要考虑的特征数量(默认为None),可以减少过拟合。
random_state: 控制随机性的种子(默认为None),确保结果的可重复性。
max_leaf_nodes: 最大叶子节点数(默认为None),限制了叶子节点的数量,可以防止过拟合。
min_impurity_decrease: 分割节点所需的最小不纯度减少量(默认为0.0),用于控制树的增长。
class_weight: 类别的权重(默认为None),用于处理不平衡数据集,可以设置为"balanced"或字典形式。
'''
# 训练模型
dt.fit(X_train, y_train)

# 示例数据的预测
example = dt.predict([[4.8, 3.0, 1.1, 0.1]])
print('示例数据预测结果:', iris.target_names[example])

# 测试集预测
y_pred = dt.predict(X_test)

# 输出准确率
print(f"测试集准确率: {accuracy_score(y_test, y_pred):.2f}")

示例数据预测结果: ['setosa']
测试集准确率: 0.98


### **随机森林**

由多棵随机生成的决策树组成的模型，大家“投票”决定结果。

In [2]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


# 加载数据
iris = load_iris()
X, y = iris.data, iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 创建随机森林模型
rf = RandomForestClassifier(n_estimators=100, criterion='gini', min_samples_split=5, max_depth=4)
'''
n_estimators: 决策树数量(默认100),值越大模型越稳定但计算量越大
criterion: 分裂标准(默认"gini"),可选"gini"或"entropy"
max_depth: 单棵树最大深度(默认None),控制模型复杂度
min_samples_split: 节点分裂最小样本数(默认2),防止过拟合
min_samples_leaf: 叶节点最小样本数(默认1),限制叶节点尺寸
max_features: 分裂时考虑的特征数(默认"sqrt"),控制随机性
bootstrap: 是否使用有放回抽样(默认True),推荐保持
oob_score: 是否使用袋外样本评估(默认False)
n_jobs: 并行任务数(默认None),-1使用全部核心
random_state: 随机种子(默认None),保证结果可复现
class_weight: 类别权重(默认None),处理不平衡数据
max_samples: 每棵树最大样本数(默认None),控制子采样
'''

# 训练模型
rf.fit(X_train, y_train)

# 示例数据的预测
example = rf.predict([[4.8, 3.0, 1.1, 0.1]])
print('示例数据预测结果:', iris.target_names[example])

# 测试集预测
y_pred = rf.predict(X_test)

# 输出准确率
print(f"测试集准确率: {accuracy_score(y_test, y_pred):.2f}")

示例数据预测结果: ['setosa']
测试集准确率: 0.98


### KNN算法(K-Nearest Neighbors)
给定一个待分类的数据点，找到在训练集中与它距离最近的K个数据点，多数邻居是什么类别，新数据就是什么类别。

算法步骤：
1. 计算测试样本与训练集中每个样本的距离，并按照距离的远近排序
2. 选取与当前测试样本最近的k个训练样本，作为该测试样本的邻居
3. 统计这k个样本的类别频次，频次最高的类别，即为测试样本的类别

In [3]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

In [4]:
# 加载数据
iris = load_iris()
X, y = iris.data, iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 创建KNN模型
knn = KNeighborsClassifier(n_neighbors=5)

# 训练模型
knn.fit(X_train, y_train)

# 示例数据的预测
example = knn.predict([[4.8, 3.0, 1.1, 0.1]])
print('示例数据预测结果:', iris.target_names[example])

# 测试集预测
y_pred = knn.predict(X_test)

# 输出准确率
print(f"测试集准确率: {accuracy_score(y_test, y_pred):.2f}")

示例数据预测结果: ['setosa']
测试集准确率: 0.98


### SVM(支持向量机)

支持向量机用于寻找两个类别的最佳决策边界。尽管有多分类问题的变形和回归任务的变形，但其最好的应用是二分类。

<img src="./assets/SVM.png" 
     alt="SVM在二维线性可分空间的示意图" 
     style="width:30%; display: block; margin: 0 auto;" />

**性能指标**：SVM要使两条边界线的间隔d取最大值，其所确定的直线在d/2处。

**支持向量**：两条边界线所插入（触碰）的向量叫支持向量。

**训练数据**：${(X_i,y_i)}$,其中$X_i$是向量，$y_i$是标签，取±1。

这样简单的模型只能解决线性可分的问题，为了解决非线性可分的问题，引入核函数，将数据映射到高维空间。

一般来说，所映射到的维度越高，其可以被一个超平面分开的概率就越大，如果映射到无穷维空间，可以被一个超平面分开的概率为1。

<img src="./assets/凸优化问题.png" 
     alt="SVM将寻找最大间距的超平面的问题转化为这样的凸优化问题" 
     style="width:50%; display: block; margin: 0 auto;" />

SVM将寻找最大间距的超平面的问题转化为这样的凸优化问题，具体的推导过程参考胡浩基老师的课程。

<img src="./assets/引入核函数和正则项.png" 
     alt="SVM引入核函数和正则项" 
     style="width:50%; display: block; margin: 0 auto;" />

引入核函数和正则项后，SVM可以解决非线性平面的问题，其中C是SVM的重要的参数。

常用的核函数有线性核，高斯核，多项式核，tanh核，一般高斯核用的比较多。

In [5]:
# 导入必要的库
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler  # SVM对数据尺度敏感，建议标准化

# 1. 加载数据
iris = datasets.load_iris()
X, y = iris.data, iris.target

# 2. 数据预处理（SVM对特征尺度敏感，必须做标准化）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. 划分训练集和测试集（保持30%测试比例与示例一致）
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.3, 
    random_state=42  # 固定随机种子保证可复现性
)

# 4. 创建SVM模型（使用RBF核，这是最常用的核函数）
svm = SVC(
    kernel='rbf',         # 径向基核函数
    C=1.0,                # 正则化参数
    gamma='scale',        # 核系数（'scale'表示自动计算）
    probability=True      # 启用概率估计（可选）
)

# 5. 训练模型
svm.fit(X_train, y_train)

# 6. 示例数据预测（注意：需要对新数据同样做标准化！）
example = svm.predict(scaler.transform([[4.8, 3.0, 1.1, 0.1]]))
print('示例数据预测结果:', iris.target_names[example])

# 7. 测试集预测
y_pred = svm.predict(X_test)

# 8. 输出准确率（保留2位小数）
print(f"测试集准确率: {accuracy_score(y_test, y_pred):.2f}")

# 可选：输出更详细的分类报告
from sklearn.metrics import classification_report
print("\n分类报告：")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

示例数据预测结果: ['setosa']
测试集准确率: 1.00

分类报告：
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      1.00      1.00        13
   virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



示例数据预测结果: ['setosa']
测试集准确率: 0.98
